In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


torch.manual_seed(0)
n = 100
x = torch.rand(n, 1)
y = torch.sin(2 * torch.pi * x) + torch.rand(n, 1)

In [16]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

class Model(nn.Module):
    def __init__(self, input_size=1, hidden_size=10, output_size=1):
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        y = self.linear1(x)
        y = F.sigmoid(y)
        y = self.linear2(y)
        return y

# 隠れ層の数を指定可能にする
class Model2(nn.Module):
    def __init__(self, input_size=1, hidden_size=10, output_size=1, num_hidden_layers=1):
        super().__init__()
        layers = []
        layers.append(nn.Linear(input_size, hidden_size))
        for _ in range(num_hidden_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
        self.hidden_layers = nn.ModuleList(layers)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        x = self.output_layer(x)
        return x


In [ ]:
import csv
import os

lr = 0.1
iters = 10000
hidden_size = 10
results_dir = 'results'
os.makedirs(results_dir, exist_ok=True)

# relu
print("loss(relu): ")

num_hidden_layers_list = [2, 3, 5, 10, 20]  # 隠れ層の数を指定するリスト

for num_hidden_layers in num_hidden_layers_list:
    model2 = Model2(hidden_size=hidden_size, num_hidden_layers=num_hidden_layers)
    optimizer2 = torch.optim.SGD(model2.parameters(), lr=lr)

    # CSV保存用
    csv_path = os.path.join(results_dir, f'06c_loss_hidden{num_hidden_layers}_hidSz{hidden_size}_lr{lr}.csv')
    loss_records = []

    for i in range(iters):
        y_pred2 = model2(x)
        loss2 = F.mse_loss(y, y_pred2)
        
        loss2.backward()
        optimizer2.step()
        optimizer2.zero_grad()
        
        if i % 1000 == 0:
            print(loss2.item())
            loss_records.append([num_hidden_layers, lr, hidden_size, i, loss2.item()])

    # 最終iterationのlossも記録（最後のi%1000!=0の場合）
    if (iters - 1) % 1000 != 0:
        loss_records.append([num_hidden_layers, lr, hidden_size, iters - 1, loss2.item()])

    # CSV書き出し
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['num_hidden_layers', 'lr', 'hidden_size', 'iteration', 'loss'])
        writer.writerows(loss_records)
    print(f'  -> saved: {csv_path}')

    # 学習後のモデルの出力を可視化
    plt.scatter(x.numpy(), y.numpy(), s=10, label='training data')
    x_test = torch.linspace(0, 1, 300).reshape(-1, 1)
    model2.eval()
    with torch.no_grad():
        y_test2 = model2(x_test)
        

    x_test2 = torch.linspace(0, 1, 300).reshape(-1, 1)
    model2.eval()
    with torch.no_grad():
        y_test2 = model2(x_test2)

    # 曲線として描画
    # plt.plot(x_test.numpy(), y_test.numpy(), label='by sigmoid', color = 'red')
    plt.plot(x_test2.numpy(), y_test2.numpy(), label=f'by relu (hidden layers: {len(model2.hidden_layers)})', color = 'blue')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f'Toy Dataset (n={n})\n y = sin(2πx) + ε')
    plt.legend()
    plt.show()

In [ ]:
## 各隠れ層の活性化（ReLU通過後）のヒストグラムを描画
## weight_init_activation_histogram.py のスタイル

import numpy as np

torch.manual_seed(0)
input_data = torch.rand(1000, 1)  # toy dataset と同じ入力次元(1)

for num_hidden in [2, 20]:
    hidden_size = 10
    model = Model2(hidden_size=hidden_size, num_hidden_layers=num_hidden)
    
    # forward しながら各隠れ層の activation を記録
    activations = {}
    h = input_data
    for i, layer in enumerate(model.hidden_layers):
        h = F.relu(layer(h))
        activations[i] = h.detach().numpy()
    
    # CSV保存: 各層の全ニューロンのraw activation値
    csv_path = os.path.join(results_dir, f'06c_activations_hidden{num_hidden}_hidSz{hidden_size}.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['num_hidden_layers', 'hidden_size', 'layer', 'neuron_index', 'value'])
        for layer_idx, act in activations.items():
            # act shape: (1000, hidden_size)
            for neuron_idx in range(act.shape[1]):
                for val in act[:, neuron_idx]:
                    writer.writerow([num_hidden, hidden_size, layer_idx, neuron_idx, val])
    print(f'saved: {csv_path}')
    
    # ヒストグラム描画
    n_layers = len(activations)
    fig, axes = plt.subplots(1, n_layers, figsize=(min(n_layers * 2.5, 30), 3))
    if n_layers == 1:
        axes = [axes]
    fig.suptitle(f'Activation histogram (num_hidden_layers={num_hidden}, hidden_size={hidden_size})\nInitial weights (before training)', fontsize=12)
    
    for i, ax in enumerate(axes):
        data = activations[i].flatten()
        ax.hist(data, 30, range=(0, data.max() + 0.01) if data.max() > 0 else (0, 1))
        ax.set_title(f'Layer {i+1}')
        if i != 0:
            ax.set_yticks([])
        # 0の割合を表示
        zero_ratio = np.sum(data == 0) / len(data) * 100
        ax.text(0.95, 0.95, f'{zero_ratio:.0f}% dead', transform=ax.transAxes,
                ha='right', va='top', fontsize=8, color='red')
    
    plt.tight_layout()
    plt.show()

In [ ]:
## CSVから読み込んで分析

import pandas as pd
import glob as glob_mod

results_dir = 'results'

# --- 1. 学習曲線比較: 全 num_hidden_layers の loss 推移を1枚に重ねてプロット ---
loss_files = sorted(glob_mod.glob(os.path.join(results_dir, '06c_loss_hidden*.csv')))
print(f'Loss CSVs found: {len(loss_files)}')

plt.figure(figsize=(8, 5))
for fpath in loss_files:
    df = pd.read_csv(fpath)
    label = f'hidden_layers={df["num_hidden_layers"].iloc[0]}'
    plt.plot(df['iteration'], df['loss'], marker='o', markersize=3, label=label)

plt.xlabel('Iteration')
plt.ylabel('Loss (MSE)')
plt.title('Training Loss Comparison by Number of Hidden Layers')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- 2. 活性化ヒストグラム再描画: CSVからraw値を読み込んで描画 ---
act_files = sorted(glob_mod.glob(os.path.join(results_dir, '06c_activations_hidden*.csv')))
print(f'Activation CSVs found: {len(act_files)}')

for fpath in act_files:
    df = pd.read_csv(fpath)
    num_hidden = df['num_hidden_layers'].iloc[0]
    hidden_size = df['hidden_size'].iloc[0]
    layers = sorted(df['layer'].unique())
    n_layers = len(layers)
    
    fig, axes = plt.subplots(1, n_layers, figsize=(min(n_layers * 2.5, 30), 3))
    if n_layers == 1:
        axes = [axes]
    fig.suptitle(f'Activation histogram (num_hidden_layers={num_hidden}, hidden_size={hidden_size})\n[from CSV]', fontsize=12)
    
    for idx, layer_id in enumerate(layers):
        ax = axes[idx]
        data = df[df['layer'] == layer_id]['value'].values
        ax.hist(data, 30, range=(0, data.max() + 0.01) if data.max() > 0 else (0, 1))
        ax.set_title(f'Layer {layer_id + 1}')
        if idx != 0:
            ax.set_yticks([])
        zero_ratio = np.sum(data == 0) / len(data) * 100
        ax.text(0.95, 0.95, f'{zero_ratio:.0f}% dead', transform=ax.transAxes,
                ha='right', va='top', fontsize=8, color='red')
    
    plt.tight_layout()
    plt.show()